# 05 · EGARCH-LSTM and the HAR-EGARCH-LSTM ablation

Same network, training rules and seeds as notebook 04; only the inputs change.

* **EGARCH-LSTM** – inputs per day: log EGARCH conditional volatility and the EGARCH standardized shock
  (from notebook 03, estimated on 2016–2022 only). It sees *daily-return* information only.
* **HAR-EGARCH-LSTM** (ablation) – HAR inputs and EGARCH inputs together. If it does not beat
  HAR-RV-LSTM, the EGARCH information adds nothing once the intraday RV history is known.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import json
import pandas as pd
import torch
from volforecast import config as C, lstm
from volforecast.evaluation import metric_table

torch.set_num_threads(max(1, torch.get_num_threads()))
panel = pd.read_parquet(C.PROCESSED_DIR / "daily_panel.parquet")
panel = panel.merge(pd.read_parquet(C.PROCESSED_DIR / "egarch_features.parquet"),
                    on=["Symbol", "Date"], how="inner")
universe = pd.read_csv(C.PROCESSED_DIR / "universe.csv")["Symbol"].tolist()

## EGARCH-LSTM

In [2]:
NAME = "EGARCH-LSTM"
out, history, models, info = lstm.run_lstm(panel, NAME, universe, verbose=True)
print(info)

seed 11 | epoch  1 | train 0.2791 | valid 0.2975


seed 11 | epoch  2 | train 0.2407 | valid 0.2786


seed 11 | epoch  3 | train 0.2360 | valid 0.2946


seed 11 | epoch  4 | train 0.2347 | valid 0.2828


seed 11 | epoch  5 | train 0.2325 | valid 0.2798


seed 11 | epoch  6 | train 0.2323 | valid 0.2793


... (57 outputs left out)

seed 33 | epoch 15 | train 0.2269 | valid 0.2737


seed 33 | epoch 16 | train 0.2268 | valid 0.2776


seed 33 | epoch 17 | train 0.2269 | valid 0.2803


{'n_train': 152935, 'n_valid': 46599, 'n_test': 29774, 'scaler_mean': [0.7314979685065346, 0.018716352703294054], 'scaler_std': [0.3141091498348045, 0.995029481926363]}


In [3]:
slug = NAME.lower().replace("-", "_")
for split in ("valid", "test"):
    out[split].to_csv(C.FORECAST_DIR / f"{slug}_{split}_forecasts.csv", index=False)
history.to_csv(C.TABLE_DIR / f"{slug}_training_history.csv", index=False)
for seed, m in models.items():
    torch.save(m.state_dict(), C.MODEL_DIR / f"{slug}_seed{seed}.pt")
json.dump(info, open(C.MODEL_DIR / f"{slug}_info.json", "w"), indent=1)

seed_cols = [c for c in out["test"] if "_seed" in c]
for split in ("valid", "test"):
    print(split.upper())
    display(metric_table(out[split], seed_cols + [NAME]))

VALID


,Obs,MSE,MAE,QLIKE
Model,,,,
EGARCH-LSTM_seed11,46599,5.683660e-07,0.000227,0.271138
EGARCH-LSTM_seed22,46599,5.562060e-07,0.000226,0.269628
EGARCH-LSTM_seed33,46599,5.931840e-07,0.000224,0.271797
EGARCH-LSTM,46599,5.710462e-07,0.000225,0.270273


TEST


,Obs,MSE,MAE,QLIKE
Model,,,,
EGARCH-LSTM_seed11,29774,1.232800e-07,0.000206,0.272016
EGARCH-LSTM_seed22,29774,1.228224e-07,0.000206,0.271453
EGARCH-LSTM_seed33,29774,1.117172e-07,0.000199,0.270325
EGARCH-LSTM,29774,1.185555e-07,0.000203,0.270853


## HAR-EGARCH-LSTM

In [4]:
NAME = "HAR-EGARCH-LSTM"
out, history, models, info = lstm.run_lstm(panel, NAME, universe, verbose=True)
print(info)

seed 11 | epoch  1 | train 0.2275 | valid 0.2255


seed 11 | epoch  2 | train 0.2049 | valid 0.2259


seed 11 | epoch  3 | train 0.2033 | valid 0.2276


seed 11 | epoch  4 | train 0.2019 | valid 0.2253


seed 11 | epoch  5 | train 0.2013 | valid 0.2194


seed 11 | epoch  6 | train 0.2005 | valid 0.2276


... (47 outputs left out)

seed 33 | epoch 14 | train 0.1963 | valid 0.2260


seed 33 | epoch 15 | train 0.1968 | valid 0.2220


seed 33 | epoch 16 | train 0.1963 | valid 0.2258


{'n_train': 152343, 'n_valid': 46599, 'n_test': 29774, 'scaler_mean': [-8.08704153330432, -7.973768953701799, -7.895593850975472, 0.730930130780369, 0.018718209390786382], 'scaler_std': [0.8761357882524118, 0.7730110064691118, 0.6949768144960171, 0.3137162779317254, 0.994812132021456]}


In [5]:
slug = NAME.lower().replace("-", "_")
for split in ("valid", "test"):
    out[split].to_csv(C.FORECAST_DIR / f"{slug}_{split}_forecasts.csv", index=False)
history.to_csv(C.TABLE_DIR / f"{slug}_training_history.csv", index=False)
for seed, m in models.items():
    torch.save(m.state_dict(), C.MODEL_DIR / f"{slug}_seed{seed}.pt")
json.dump(info, open(C.MODEL_DIR / f"{slug}_info.json", "w"), indent=1)

seed_cols = [c for c in out["test"] if "_seed" in c]
for split in ("valid", "test"):
    print(split.upper())
    display(metric_table(out[split], seed_cols + [NAME]))

VALID


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-EGARCH-LSTM_seed11,46599,5.140544e-07,0.000190,0.218766
HAR-EGARCH-LSTM_seed22,46599,5.139334e-07,0.000190,0.218279
HAR-EGARCH-LSTM_seed33,46599,5.237481e-07,0.000188,0.219515
HAR-EGARCH-LSTM,46599,5.161232e-07,0.000189,0.218364


TEST


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-EGARCH-LSTM_seed11,29774,8.724177e-08,0.000146,0.194617
HAR-EGARCH-LSTM_seed22,29774,8.739249e-08,0.000146,0.194243
HAR-EGARCH-LSTM_seed33,29774,8.705440e-08,0.000142,0.191561
HAR-EGARCH-LSTM,29774,8.703029e-08,0.000144,0.193106
